# 04 LoRA MVP Bootstrap Training Colab

本 notebook 专门执行正式 LoRA MVP bootstrap 训练。`03_train_lora_colab.ipynb` 保持用于模块探测和 20 step smoke training；本 notebook 从已经提交的 `data/jsonl/lora_mvp_train.local.jsonl` 开始，先 preflight，再跑第一版 600 step 训练。

目标：产出 `checkpoints/qwen3-asr-1.7b-lora-mvp/`，用于下一步 LoRA adapter 推理和 MVP 150 held-out test 对比。


In [ ]:
# 挂载 Google Drive。
# 项目目录默认使用 /content/drive/MyDrive/qwen3-asr。
from google.colab import drive

drive.mount('/content/drive')


In [ ]:
# 安装最小依赖。
# qwen-asr 0.0.6 当前固定依赖 transformers==4.57.6 和 accelerate==1.12.0。
# 当前训练不依赖 torchao。部分 Colab runtime 预装 torchao==0.10.0，
# 会导致 PEFT 在 get_peft_model() 阶段报版本不兼容，因此这里主动卸载。
%pip -q install --upgrade --upgrade-strategy only-if-needed qwen-asr==0.0.6 transformers==4.57.6 accelerate==1.12.0 peft bitsandbytes huggingface_hub pyyaml
%pip -q install pandas==2.2.2 requests==2.32.4
%pip -q uninstall -y torchao


In [ ]:
# 读取项目路径和 MVP 训练配置。
from pathlib import Path
import json
import yaml

PROJECT_DIR = Path('/content/drive/MyDrive/qwen3-asr')
CONFIG_PATH = PROJECT_DIR / 'configs/train/qwen3_asr_lora_mvp_train.yaml'
TRAIN_MANIFEST = PROJECT_DIR / 'data/jsonl/lora_mvp_train.local.jsonl'
VAL_MANIFEST = PROJECT_DIR / 'data/jsonl/lora_mvp_val.local.jsonl'
TRAIN_OUTPUT_DIR = PROJECT_DIR / 'checkpoints/qwen3-asr-1.7b-lora-mvp'
MODEL_ID = 'Qwen/Qwen3-ASR-1.7B'
DTYPE = 'float16'
DEVICE_MAP = 'cuda:0'
MAX_STEPS = 600

assert PROJECT_DIR.exists(), f'项目目录不存在: {PROJECT_DIR}'
assert CONFIG_PATH.exists(), f'缺少训练配置: {CONFIG_PATH}'
assert TRAIN_MANIFEST.exists(), f'缺少 train manifest: {TRAIN_MANIFEST}'
assert VAL_MANIFEST.exists(), f'缺少 val manifest: {VAL_MANIFEST}'

config = yaml.safe_load(CONFIG_PATH.read_text(encoding='utf-8'))
print('PROJECT_DIR =', PROJECT_DIR)
print('CONFIG_PATH =', CONFIG_PATH)
print('TRAIN_MANIFEST =', TRAIN_MANIFEST)
print('VAL_MANIFEST =', VAL_MANIFEST)
print('TRAIN_OUTPUT_DIR =', TRAIN_OUTPUT_DIR)
print('config model =', config.get('model', {}))
print('config data =', config.get('data', {}))
print('config training =', config.get('training', {}))


In [ ]:
# 检查 GPU。
# 如果这里没有 CUDA，先在 Colab 菜单中选择 Runtime -> Change runtime type -> GPU。
import torch

print('torch =', torch.__version__)
print('cuda available =', torch.cuda.is_available())
if torch.cuda.is_available():
    print('gpu =', torch.cuda.get_device_name(0))
    print('capability =', torch.cuda.get_device_capability(0))
else:
    raise RuntimeError('当前 runtime 没有 CUDA GPU，无法训练 Qwen3-ASR LoRA。')


## Hugging Face 登录

如果模型下载遇到权限或限流问题，先执行下面单元登录 Hugging Face。已经登录或模型可直接下载时，可以跳过。


In [ ]:
# 可选：登录 Hugging Face。
# from huggingface_hub import notebook_login
# notebook_login()


In [ ]:
# 检查 train/val manifest 和音频路径。
# 这里不加载模型，先快速确认数据完整，避免浪费 GPU 显存和下载时间。
from collections import Counter

def read_jsonl(path):
    return [json.loads(line) for line in path.read_text(encoding='utf-8').splitlines() if line.strip()]

def resolve_audio(audio):
    p = Path(audio)
    if p.is_absolute():
        return p
    return PROJECT_DIR / p

train_rows = read_jsonl(TRAIN_MANIFEST)
val_rows = read_jsonl(VAL_MANIFEST)

for name, rows in [('train', train_rows), ('val', val_rows)]:
    counts = Counter(row.get('scenario', '') for row in rows)
    missing = [str(resolve_audio(row['audio'])) for row in rows if not resolve_audio(row['audio']).exists()]
    print(name, 'rows =', len(rows), 'counts =', dict(counts), 'missing =', len(missing))
    if missing:
        print('
'.join(missing[:20]))
        raise FileNotFoundError(f'{name} manifest 有缺失音频: {len(missing)}')

train_ids = {row['base_utterance_id'] for row in train_rows}
val_ids = {row['base_utterance_id'] for row in val_rows}
overlap = train_ids & val_ids
print('train_val_base_utterance_overlap =', len(overlap))
if overlap:
    raise RuntimeError(f'train/val base_utterance_id overlap: {sorted(overlap)[:5]}')


In [ ]:
# 命令执行工具：打印 stdout/stderr tail，避免只看到 CalledProcessError。
import subprocess
import sys


def run_training_cmd(cmd, stderr_tail=12000, stdout_tail=8000):
    print('运行命令:')
    print(' '.join(map(str, cmd)))
    result = subprocess.run(
        cmd,
        cwd=str(PROJECT_DIR),
        text=True,
        capture_output=True,
    )
    print('returncode =', result.returncode)
    if result.stdout:
        print('--- stdout tail ---')
        print(result.stdout[-stdout_tail:])
    if result.stderr:
        print('--- stderr tail ---')
        print(result.stderr[-stderr_tail:])
    result.check_returncode()
    return result


In [ ]:
# Preflight：加载模型、匹配 99 个 LoRA target、写出 target/training_config，然后停止。
# 如果这里失败，不要跑正式训练；先看 stdout/stderr tail。
preflight_cmd = [
    sys.executable,
    'train/train_qwen3_asr_lora.py',
    '--config', 'configs/train/qwen3_asr_lora_mvp_train.yaml',
    '--manifest', str(TRAIN_MANIFEST),
    '--audio-root', str(PROJECT_DIR),
    '--output-dir', str(TRAIN_OUTPUT_DIR),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--quantization', config.get('model', {}).get('quantization', '4bit'),
    '--language', 'English',
    '--preflight-only',
]

run_training_cmd(preflight_cmd)


In [ ]:
# 检查 preflight 产物。
for name in ['target_modules.json', 'training_config.json', 'summary.json']:
    path = TRAIN_OUTPUT_DIR / name
    print('
===', name, '===')
    assert path.exists(), f'缺少 preflight 输出: {path}'
    text = path.read_text(encoding='utf-8')
    print(text[:4000])

summary = json.loads((TRAIN_OUTPUT_DIR / 'summary.json').read_text(encoding='utf-8'))
assert summary.get('status') == 'preflight_ok', summary
assert summary.get('count') == 99, summary.get('count')
print('
preflight 验收通过。')


In [ ]:
# 正式 LoRA MVP bootstrap 训练：默认 600 step。
# 产物会覆盖同一 output dir 中的 summary/loss_log，并保存 adapter/processor。
train_cmd = [
    sys.executable,
    'train/train_qwen3_asr_lora.py',
    '--config', 'configs/train/qwen3_asr_lora_mvp_train.yaml',
    '--manifest', str(TRAIN_MANIFEST),
    '--audio-root', str(PROJECT_DIR),
    '--output-dir', str(TRAIN_OUTPUT_DIR),
    '--model-id', MODEL_ID,
    '--dtype', DTYPE,
    '--device-map', DEVICE_MAP,
    '--quantization', config.get('model', {}).get('quantization', '4bit'),
    '--language', 'English',
    '--max-steps', str(MAX_STEPS),
]

run_training_cmd(train_cmd, stderr_tail=16000, stdout_tail=12000)


In [ ]:
# 预览训练结果。
summary_path = TRAIN_OUTPUT_DIR / 'summary.json'
loss_path = TRAIN_OUTPUT_DIR / 'loss_log.jsonl'
adapter_dir = TRAIN_OUTPUT_DIR / 'adapter'
processor_dir = TRAIN_OUTPUT_DIR / 'processor'

summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(summary, ensure_ascii=False, indent=2)[:6000])

loss_lines = [line for line in loss_path.read_text(encoding='utf-8').splitlines() if line.strip()]
print('
loss lines =', len(loss_lines))
print('
最后 20 行 loss:')
print('
'.join(loss_lines[-20:]))

assert summary.get('status') == 'trained', summary
assert summary.get('steps') == MAX_STEPS, summary.get('steps')
assert adapter_dir.exists(), f'缺少 adapter dir: {adapter_dir}'
assert processor_dir.exists(), f'缺少 processor dir: {processor_dir}'
assert len(loss_lines) == MAX_STEPS, len(loss_lines)
print('
LoRA MVP bootstrap training 完成。下一步：实现/运行 LoRA adapter 推理并在 MVP 150 held-out test 上评测。')


## 完成标准

- `summary.json` 中 `status=trained`。
- `loss_log.jsonl` 行数等于 `MAX_STEPS`，默认 600。
- `target_modules.json` 中 target count 等于 99。
- `adapter/` 和 `processor/` 已保存。
- 训练结束后不要直接宣称效果提升；下一步必须跑 LoRA adapter 推理，并在固定 MVP 150 held-out test 上与 base WER 对比。
